# Training a Potential: how to setup data pipeline, model and trainer 

# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp

## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data. 

In [1]:
%load_ext autoreload
%autoreload 2

import logging, sys
logging.basicConfig(level=logging.DEBUG)

import molpot as mpot
import torch
from molpot import alias

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [25]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cpu",
    preprocess=[
        mpot.pipline.NeighborList(cutoff=6.0)
    ],
)
rmd17_ds.prepare_data()

INFO:molpot:Done.
INFO:molpot:Extracting data...
INFO:molpot:Parsing molecule aspirin


[Frame(
     fields={
         atoms: TensorDict(
             fields={
                 R: Tensor(shape=torch.Size([21, 3]), device=cpu, dtype=torch.float32, is_shared=False),
                 Z: Tensor(shape=torch.Size([21]), device=cpu, dtype=torch.int32, is_shared=False)},
             batch_size=torch.Size([]),
             device=None,
             is_shared=False),
         cell: Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.float32, is_shared=False),
         labels: TensorDict(
             fields={
                 energy: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False),
                 forces: Tensor(shape=torch.Size([21, 3]), device=cpu, dtype=torch.float32, is_shared=False)},
             batch_size=torch.Size([]),
             device=None,
             is_shared=False),
         pairs: TensorDict(
             fields={
                 diff: Tensor(shape=torch.Size([210, 3]), device=cpu, dtype=torch.float32, is_shared=False),
     

In [26]:
from molpot.utils.locality import get_neighbor_pairs
from molpot import alias, Config
tensordict = rmd17_ds.frames[2]
cell = tensordict[alias.cell]
xyz = tensordict[alias.xyz]

neighbors, deltas, distances, number_found_pairs = get_neighbor_pairs(
    xyz, 12.0, box_vectors=None
)
neighbors[0].to(torch.int64)

tensor([ 1,  2,  2,  3,  3,  3,  4,  4,  4,  4,  5,  5,  5,  5,  5,  6,  6,  6,
         6,  6,  6,  7,  7,  7,  7,  7,  7,  7,  8,  8,  8,  8,  8,  8,  8,  8,
         9,  9,  9,  9,  9,  9,  9,  9,  9, 10, 10, 10, 10, 10, 10, 10, 10, 10,
        10, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 12,
        12, 12, 12, 12, 12, 12, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13,
        13, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 15, 15, 15,
        15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 16, 16, 16, 16, 16, 16,
        16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 17, 17, 17, 17, 17, 17, 17, 17,
        17, 17, 17, 17, 17, 17, 17, 17, 17, 18, 18, 18, 18, 18, 18, 18, 18, 18,
        18, 18, 18, 18, 18, 18, 18, 18, 18, 19, 19, 19, 19, 19, 19, 19, 19, 19,
        19, 19, 19, 19, 19, 19, 19, 19, 19, 19, 20, 20, 20, 20, 20, 20, 20, 20,
        20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20])

In [27]:
data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

number of data: 1000

                 dataset: rmd17-aspirin                 
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ label  ┃      type       ┃    unit    ┃   comment    ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ energy │ <class 'float'> │  kcal/mol  │ total energy │
│ forces │ <class 'float'> │ kcal/mol/A │  all forces  │
└────────┴─────────────────┴────────────┴──────────────┘

In [28]:
train_ds, valid_ds = torch.utils.data.random_split(rmd17_ds, [.9, .1])

train_dl = mpot.DataLoader(
    dataset=train_ds,
    batch_size=5,
    shuffle=False,
)
eval_dl = mpot.DataLoader(
    dataset=valid_ds,
    batch_size=5,
    shuffle=False,
)

## Setting up the model

In [29]:
pinet = mpot.potential.nnp.PiNet(
    depth=5,
    basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 4.0),
    cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(4.0),
    pp_nodes=[64, 64],
    pi_nodes=[64, 64],
    ii_nodes=[64, 64],
    activation=torch.nn.Tanh(),
)
readout = mpot.potential.nnp.readout.Atomwise(
    n_in=64,
    n_out=1,
    from_key=("pinet", "p1"),
    to_key=("predicts", "energy")
)
model = mpot.potential.PotentialSeq("pinet", pinet, readout)

In [30]:
from ignite.handlers import (
    Checkpoint,
    global_step_from_engine,
    TensorboardLogger,
    ProgressBar,
)
from ignite.engine import Events, create_supervised_trainer
from ignite.metrics import MeanAbsoluteError, Loss, BatchWise, EpochWise
from pathlib import Path

loss_fn = mpot.loss(torch.nn.MSELoss().to("cuda"), ("energy"), ("energy"))

trainer = mpot.PotentialTrainer(
    model,
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
    loss_fn=loss_fn,
    device="cpu",
)
trainer.add_evaluator(eval_dl, max_epochs=2, epoch_length=10)

def output_transform(data):
    return (data[0]["energy"], data[1]["energy"])

trainer.add_metric(
    "mae",
    MeanAbsoluteError(output_transform=output_transform),
    target="all",
    usage=BatchWise(),
)

trainer.attach_progressbar()
trainer.attach_tensorboard(log_dir=Path("rmd17_log"))
# basic_profiler = trainer.attach_basic_time_profiler()
# handler_profiler = trainer.attach_handlers_time_profiler()

# trainer.run_tensorboard_profiler(train_dl, log_dir=Path("rmd17_profile"))

# state = trainer.run(train_dl, max_epochs=2, epoch_length=10)
# basic_profiler.print_results(basic_profiler.get_results())
# handler_profiler.print_results(handler_profiler.get_results())
# state

DEBUG:ignite.engine.engine.Engine:Added handler for event Events.EPOCH_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_STARTED


DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_STARTED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.EPOCH_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.EPOCH_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_COMPLETED
DEBUG:ignite.engine.engine.Engine:Added handler for event Events.ITERATION_

In [73]:
trainer.run(train_dl, 10)

INFO:ignite.engine.engine.Engine:Engine run resuming from iteration 0, epoch 9 until 10 epochs
DEBUG:ignite.engine.engine.Engine:9 | 0, Firing handlers for event Events.STARTED
DEBUG:ignite.engine.engine.Engine:10 | 0, Firing handlers for event Events.EPOCH_STARTED
DEBUG:ignite.engine.engine.Engine:10 | 0, Firing handlers for event Events.GET_BATCH_STARTED
ERROR:ignite.engine.engine.Engine:Current run is terminating due to exception: concat(): argument 'tensors' (position 1) must be tuple of Tensors, not Tensor
ERROR:ignite.engine.engine.Engine:Engine run is terminating due to exception: concat(): argument 'tensors' (position 1) must be tuple of Tensors, not Tensor


TypeError: concat(): argument 'tensors' (position 1) must be tuple of Tensors, not Tensor

In [62]:
frames = rmd17_ds.frames[:10]

In [63]:
frames

[Frame(
     fields={
         atoms: TensorDict(
             fields={
                 R: Tensor(shape=torch.Size([21, 3]), device=cpu, dtype=torch.float32, is_shared=False),
                 Z: Tensor(shape=torch.Size([21]), device=cpu, dtype=torch.int32, is_shared=False)},
             batch_size=torch.Size([]),
             device=None,
             is_shared=False),
         cell: Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.float32, is_shared=False),
         labels: TensorDict(
             fields={
                 energy: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False),
                 forces: Tensor(shape=torch.Size([21, 3]), device=cpu, dtype=torch.float32, is_shared=False)},
             batch_size=torch.Size([]),
             device=None,
             is_shared=False),
         pairs: TensorDict(
             fields={
                 diff: Tensor(shape=torch.Size([210, 3]), device=cpu, dtype=torch.float32, is_shared=False),
     

In [64]:
mpot.Frame.maybe_dense_stack(frames)

TensorDict(
    fields={
        atoms: TensorDict(
            fields={
                R: Tensor(shape=torch.Size([10, 21, 3]), device=cpu, dtype=torch.float32, is_shared=False),
                Z: Tensor(shape=torch.Size([10, 21]), device=cpu, dtype=torch.int32, is_shared=False)},
            batch_size=torch.Size([10]),
            device=None,
            is_shared=False),
        cell: Tensor(shape=torch.Size([10, 3]), device=cpu, dtype=torch.float32, is_shared=False),
        labels: TensorDict(
            fields={
                energy: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                forces: Tensor(shape=torch.Size([10, 21, 3]), device=cpu, dtype=torch.float32, is_shared=False)},
            batch_size=torch.Size([10]),
            device=None,
            is_shared=False),
        pairs: LazyStackedTensorDict(
            fields={
                diff: Tensor(shape=torch.Size([10, 210, 3]), device=cpu, dtype=torch.float32,

In [65]:
mpot.Frame.maybe_dense_stack(frames)['pairs', 'i']

RuntimeError: Found more than one unique shape in the tensors to be stacked ({torch.Size([182]), torch.Size([178]), torch.Size([181]), torch.Size([177]), torch.Size([174])}). This is likely due to a modification of one of the stacked TensorDicts, where a key has been updated/created with an uncompatible shape. If the entries are intended to have a different shape, use the get_nestedtensor method instead.